# GraphRAG: Entity-Aware Retrieval

Standard vector search retrieves documents that are semantically close to the query but has no concept of relationships between entities. When a user asks "Which companies partnered with the organization that acquired Acme Corp?", flat retrieval can find documents mentioning Acme Corp's acquisition — but it cannot traverse the graph to find the acquiring organization's other partnerships. GraphRAG solves this by building a knowledge graph during ingestion and using it to answer multi-hop questions that require following chains of entity relationships.

**What you'll build:** An ingestion pipeline that extracts named entities and relationships from documents, stores them in a graph, and a query pipeline that routes entity-centric questions through graph traversal rather than (or in addition to) vector search.

**Time:** ~30 min | **Difficulty:** Advanced

## Prerequisites

- Completed the [Basic RAG](https://synapsekit.github.io/synapsekit-docs/guides/rag/) guide
- Understanding of graph data structures (nodes, edges, traversal)
- `OPENAI_API_KEY` set in your environment

## What you'll learn

- How `GraphRAGPipeline` extracts entities and relationships during ingestion
- How to configure entity types and relationship schemas
- How multi-hop queries traverse the graph to retrieve context across documents
- When GraphRAG outperforms and underperforms standard vector retrieval

In [ ]:
!pip install synapsekit -q

## Environment Setup

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-..."  # set your key here or via environment

## Step 1: Install and configure

In [ ]:
import asyncio
import os

from synapsekit.llms.openai import OpenAILLM
from synapsekit.embeddings.openai import OpenAIEmbeddings
from synapsekit.vectorstores.memory import InMemoryVectorStore
from synapsekit.pipelines import GraphRAGPipeline

llm = OpenAILLM(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = InMemoryVectorStore(embeddings=embeddings)

## Step 2: Define your entity schema

Constraining the entity types prevents the extractor from producing a noisy, unconstrained graph. Only define types that are meaningful for your domain and that users will actually query.

In [ ]:
entity_schema = {
    "entity_types": ["PERSON", "ORGANIZATION", "PRODUCT", "TECHNOLOGY", "LOCATION"],
    "relationship_types": [
        "WORKS_AT", "FOUNDED", "ACQUIRED", "PARTNERED_WITH",
        "BUILT_ON", "COMPETES_WITH", "LOCATED_IN",
    ],
}
print("Entity schema defined:", entity_schema)

## Step 3: Initialize the GraphRAG pipeline

In [ ]:
pipeline = GraphRAGPipeline(
    llm=llm,
    embeddings=embeddings,
    vectorstore=vectorstore,
    entity_schema=entity_schema,
    max_hops=2,          # how deep to traverse from a matched entity
    top_k_vector=5,      # fallback vector results for non-entity queries
    community_summary=True,  # summarize dense entity clusters during ingestion
)
print("GraphRAGPipeline initialized.")

## Step 4: Ingest documents

During ingestion, `GraphRAGPipeline` runs two passes: vector embedding (same as standard RAG) and entity/relationship extraction. The extraction pass is LLM-driven and proportional to document volume.

In [ ]:
docs = [
    "OpenAI was founded by Sam Altman, Greg Brockman, and Elon Musk in 2015. "
    "OpenAI developed GPT-4, which is built on transformer architecture.",

    "Microsoft acquired a major stake in OpenAI in 2019 and again in 2023. "
    "Microsoft integrated GPT-4 into its Copilot product suite.",

    "Anthropic was founded by Dario Amodei and Daniela Amodei, former OpenAI researchers. "
    "Anthropic built Claude, which competes with GPT-4.",

    "Google DeepMind partnered with Google Cloud to deploy Gemini, "
    "a multimodal model that competes with both GPT-4 and Claude.",

    "Sam Altman previously worked at Y Combinator before co-founding OpenAI. "
    "Y Combinator is located in San Francisco.",
]

async def ingest():
    result = await pipeline.aadd(docs)
    print(f"Ingested {result.num_documents} documents")
    print(f"Extracted {result.num_entities} entities, {result.num_relationships} relationships")

asyncio.run(ingest())

## Step 5: Inspect the graph

Before querying, verify that the graph contains the entities and relationships you expect. Missing relationships are almost always a sign that the entity schema needs adjustment.

In [ ]:
async def inspect_graph():
    graph = await pipeline.aget_graph()

    print("Entities:")
    for entity in graph.entities[:10]:
        print(f"  [{entity.type}] {entity.name}")

    print("\nRelationships:")
    for rel in graph.relationships[:10]:
        print(f"  {rel.source} --[{rel.type}]--> {rel.target}")

asyncio.run(inspect_graph())

## Step 6: Run a single-hop query

In [ ]:
async def single_hop():
    result = await pipeline.aquery("What did Microsoft build using GPT-4?")
    print(result.answer)
    print("\nSources:", [s.text[:60] for s in result.sources])

asyncio.run(single_hop())

## Step 7: Run a multi-hop query

Multi-hop queries require the pipeline to traverse relationships. The traversal starts from entities matched in the query, follows edges up to `max_hops` deep, and collects the text of all reachable document nodes.

In [ ]:
async def multi_hop():
    # Answering this requires: query -> OpenAI -> founded_by -> Sam Altman -> works_at -> Y Combinator -> located_in -> San Francisco
    result = await pipeline.aquery(
        "Where was Sam Altman based before he co-founded OpenAI?"
    )
    print(result.answer)
    print("\nTraversal path:", result.traversal_path)

asyncio.run(multi_hop())

## Step 8: Stream a complex analytical question

In [ ]:
async def stream_analysis():
    question = "Which organizations are competing with OpenAI, and what products are involved?"
    print(f"Q: {question}\n")
    async for chunk in pipeline.astream(question):
        print(chunk, end="", flush=True)
    print()

asyncio.run(stream_analysis())

## Complete Working Example

A single self-contained cell that runs the entire GraphRAG pipeline end-to-end.

In [ ]:
import asyncio
import os

from synapsekit.llms.openai import OpenAILLM
from synapsekit.embeddings.openai import OpenAIEmbeddings
from synapsekit.vectorstores.memory import InMemoryVectorStore
from synapsekit.pipelines import GraphRAGPipeline

DOCS = [
    "OpenAI was founded by Sam Altman, Greg Brockman, and Elon Musk in 2015. "
    "OpenAI developed GPT-4, which is built on transformer architecture.",
    "Microsoft acquired a major stake in OpenAI in 2019 and again in 2023. "
    "Microsoft integrated GPT-4 into its Copilot product suite.",
    "Anthropic was founded by Dario Amodei and Daniela Amodei, former OpenAI researchers. "
    "Anthropic built Claude, which competes with GPT-4.",
    "Google DeepMind partnered with Google Cloud to deploy Gemini, "
    "a multimodal model that competes with both GPT-4 and Claude.",
    "Sam Altman previously worked at Y Combinator before co-founding OpenAI. "
    "Y Combinator is located in San Francisco.",
]

ENTITY_SCHEMA = {
    "entity_types": ["PERSON", "ORGANIZATION", "PRODUCT", "TECHNOLOGY", "LOCATION"],
    "relationship_types": [
        "WORKS_AT", "FOUNDED", "ACQUIRED", "PARTNERED_WITH",
        "BUILT_ON", "COMPETES_WITH", "LOCATED_IN",
    ],
}


async def main():
    llm = OpenAILLM(model="gpt-4o-mini")
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = InMemoryVectorStore(embeddings=embeddings)

    pipeline = GraphRAGPipeline(
        llm=llm,
        embeddings=embeddings,
        vectorstore=vectorstore,
        entity_schema=ENTITY_SCHEMA,
        max_hops=2,
        top_k_vector=5,
        community_summary=True,
    )

    result = await pipeline.aadd(DOCS)
    print(f"Extracted {result.num_entities} entities, {result.num_relationships} relationships\n")

    questions = [
        "What did Microsoft build using GPT-4?",
        "Where was Sam Altman based before he co-founded OpenAI?",
        "Which organizations are competing with OpenAI, and what products are involved?",
    ]

    for question in questions:
        print(f"Q: {question}")
        result = await pipeline.aquery(question)
        print(f"A: {result.answer}\n")


asyncio.run(main())

## Next Steps

- [Self-RAG](https://synapsekit.github.io/synapsekit-docs/guides/retrieval/self-rag) — add relevance grading on top of graph-retrieved context
- [Query Decomposition](https://synapsekit.github.io/synapsekit-docs/guides/retrieval/query-decomposition) — break multi-hop questions into explicit sub-queries before graph traversal
- [Adaptive RAG](https://synapsekit.github.io/synapsekit-docs/guides/retrieval/adaptive-rag) — route graph queries to a stronger model while keeping vector-only queries on a cheaper one